In [ ]:
!pip install diffusers

In [ ]:
from diffusers import UNet2DModel, DDIMScheduler
import torch

class DiffusionModel:
    def __init__(self, sample_dim, model_name, isPretrained=True, scheduler_cls=DDIMScheduler, num_inference_steps=1000):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sample_dim = sample_dim    # e.g., (1, 3, 64, 64) or (1, 3, 256, 256)
        self.model_name = model_name
        self.isPretrained = isPretrained
        self.scheduler_cls = scheduler_cls
        self.num_inference_steps = num_inference_steps  # Number of steps for the scheduler
        self.bottleneck = None  # will hold the mid block output during forward pass
        self.load_model()

    def load_model(self):
        if self.isPretrained:
            # Load the pretrained UNet model and move it to the proper device
            self.model = UNet2DModel.from_pretrained(self.model_name).to(self.device)
            # Load the scheduler (for example, DDIM scheduler)
            self.scheduler = self.scheduler_cls.from_pretrained(self.model_name, rescale_zero_terminal_snr=True)
            # Set the number of inference steps to initialize the scheduler's timesteps
            self.scheduler.set_timesteps(self.num_inference_steps)
            # Move scheduler internal tensors to the device (if necessary)
            if hasattr(self.scheduler, 'alphas_cumprod'):
                self.scheduler.alphas_cumprod = self.scheduler.alphas_cumprod.to(self.device)
            # Register a forward hook on the mid block to capture its output (bottleneck)
            self.model.mid_block.register_forward_hook(self._save_bottleneck)
        else:
            print("Non-pretrained model loading is not implemented yet.")
            exit()

    def _save_bottleneck(self, module, input, output):
        """
        Hook to capture the output (feature map) of the mid block.
        """
        self.bottleneck = output
        return output  # return the original output without modification

    def inference_step(self, x, t):
        """
        Runs a single inference step through the UNet.
        Returns the noise prediction and the captured bottleneck feature map.

        Args:
            x (torch.Tensor): Input tensor (e.g., a noisy sample)
            t (int or torch.Tensor): The timestep (if int, converted to a tensor)
        Returns:
            noise_pred (torch.Tensor): The noise prediction from the UNet
            bottleneck (torch.Tensor): The mid block's feature map for this step
        """
        # Ensure timestep is a tensor on the correct device
        if not isinstance(t, torch.Tensor):
            t_tensor = torch.tensor([t]).to(self.device)
        else:
            t_tensor = t.to(self.device)

        # Forward pass (the hook on mid_block will capture the bottleneck)
        with torch.no_grad():
            out = self.model(x, t_tensor)
        noise_pred = out.sample  # assuming the model returns a UNet2DOutput with attribute "sample"
        return noise_pred, self.bottleneck

    def diffusion_step(self, x, t):
        """
        Performs one reverse diffusion step: runs inference and then uses the scheduler
        to compute the next (denoised) sample.

        Args:
            x (torch.Tensor): Current sample (noisy image)
            t (int): Current timestep
        Returns:
            new_x (torch.Tensor): The updated sample (x_{t-1})
            bottleneck (torch.Tensor): The bottleneck feature map from the inference step
        """
        # Get noise prediction and bottleneck features
        noise_pred, bottleneck = self.inference_step(x, t)
        # Create a timestep tensor for the scheduler on the correct device
        t_tensor = torch.tensor([t]).to(self.device)
        # Compute the previous sample using the scheduler
        new_x = self.scheduler.step(noise_pred, t_tensor, x).prev_sample
        return new_x, bottleneck

# Example usage:
if __name__ == "__main__":
    import matplotlib.pyplot as plt
    from tqdm import tqdm

    # For example, using a model trained on 256x256 images.
    sample_dim = (1, 3, 256, 256)
    model_name = "google/ddpm-ema-celebahq-256"  # or any compatible pretrained DDPM model

    diffusion_model = DiffusionModel(sample_dim, model_name, isPretrained=True, num_inference_steps=250)

    def generate_brownian_noise(num_steps, shape, device="cuda", dt=1.0):
        # Sample Gaussian increments (std = sqrt(dt))
        increments = torch.randn(num_steps, *shape, device=device) * (dt ** 0.5)
        # Cumulative sum to get Brownian noise
        brownian_noise = torch.cumsum(increments, dim=0)
        return brownian_noise

    # Settings
    batch_size = 4
    sample_dim = (batch_size, 3, 256, 256)  # (B, C, H, W)
    num_steps_brownian = 250  # Number of steps to generate the Brownian trajectory

    # Assume diffusion_model is your previously defined diffusion model wrapper
    device = diffusion_model.device

    # Generate initial Gaussian noise sample for 4 images
    gaussian_noise = torch.randn(sample_dim).to(device)

    # Generate initial Brownian noise sample for 4 images:
    # For each image in the batch, generate an independent Brownian trajectory and take the final state.
    brownian_noise_list = []
    for i in range(batch_size):
        traj = generate_brownian_noise(num_steps_brownian, sample_dim[1:], device=device)
        brownian_noise_list.append(traj[-1])
    brownian_noise = torch.stack(brownian_noise_list, dim=0)  # shape: (4, 3, 256, 256)

    # --- Reverse diffusion on Gaussian noise ---
    x_gaussian = gaussian_noise.clone()
    for t in tqdm(diffusion_model.scheduler.timesteps, desc="Reverse diffusion (Gaussian)", total=len(diffusion_model.scheduler.timesteps)):
        x_gaussian, _ = diffusion_model.diffusion_step(x_gaussian, t)

    # --- Reverse diffusion on Brownian noise ---
    x_brownian = brownian_noise.clone()
    for t in tqdm(diffusion_model.scheduler.timesteps, desc="Reverse diffusion (Brownian)", total=len(diffusion_model.scheduler.timesteps)):
        x_brownian, _ = diffusion_model.diffusion_step(x_brownian, t)

    # Utility to display a grid of 4 images
    def show_images(image_tensor, title):
        # image_tensor: shape (batch_size, 3, H, W) with values in [-1,1]
        imgs = (image_tensor * 0.5 + 0.5).clamp(0,1).detach().cpu().numpy()
        fig, axes = plt.subplots(2, 2, figsize=(8, 8))
        for i, ax in enumerate(axes.flat):
            img = imgs[i].transpose(1, 2, 0)  # from (C,H,W) to (H,W,C)
            ax.imshow(img)
            ax.axis("off")
        plt.suptitle(title)
        plt.show()

    # Display the results
    show_images(x_gaussian, "Generated Images from Gaussian Noise")
    show_images(x_brownian, "Generated Images from Brownian Noise")

    # Optionally, print the shape of the bottleneck from the last step
    print("Bottleneck feature map shape:", _.shape)
